# Explorer 04 - Labels and the majors

**Purpose.** Rebuild the Universal, Sony, and Warner label trees, measure major versus independent share, and profile who dominates each label.

**Feeds main report section:** 5.3 RQ3 - Label power

**Modules used:** `vinyl.labels`, `vinyl.viz`

**Tables read:** `labels`, `release_labels`, `release_artists`, `release_credits`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest, networks

spark = ingest.get_spark(app_name="vinyl-04-labels", shuffle_partitions=16)
from vinyl import labels as label_power, viz

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
labels = ingest.read_table(spark, snapshot, "labels")
release_labels = ingest.read_table(spark, snapshot, "release_labels")
release_artists = ingest.read_table(spark, snapshot, "release_artists")
release_credits = ingest.read_table(spark, snapshot, "release_credits")

## 1. Find the root labels of the three majors

`config.MAJOR_ROOT_IDS` holds the root label ids confirmed in T04 (Universal 38404, Sony 353657, Warner 2345). Check here that these ids appear in the labels table.

In [ ]:
# The three root rows, by id.
labels.where(F.col("label_id").isin(list(config.MAJOR_ROOT_IDS.values()))).show(truncate=False)
# Every label whose name looks like a major, to spot other candidate roots.
labels.where(F.col("label_name").rlike("(?i)^(universal music|sony music|warner music)")).show(50, truncate=False)

## 2. Resolve every label to its root

In [ ]:
roots = label_power.resolve_label_roots(labels, max_depth=12).cache()
roots.groupBy("reached_root").count().show()          # False rows: raise max_depth or inspect
roots.groupBy("depth").count().orderBy("depth").show()

In [ ]:
tagged = label_power.tag_major_labels(roots)
tagged.groupBy("major").count().show()

## 3. Major versus independent share by year

In [ ]:
major_share = label_power.major_share_by_year(release_labels, tagged).toPandas().sort_values(["year", "major"])
viz.plot_lines(
    major_share, "year", "share", "major", percent=True,
    title="Share of releases on Universal, Sony, Warner and independent labels, 2016-2026 (2026 is partial: Jan-Sep)",
)

In [ ]:
# A release on labels of two groups counts in both, so the bars are rescaled to 100% of group memberships.
viz.plot_stacked_share(
    major_share, "year", "n_releases", "major",
    title="Release counts by label group, rescaled to 100% (2026 is partial: Jan-Sep)",
)
major_share[major_share.year.isin([2016, 2025])]

## 4. Artist concentration per label

In [ ]:
real_artists = release_artists.where(~F.col("artist_id").isin(list(networks.PLACEHOLDER_ARTIST_IDS)))  # drop "Various", "Unknown Artist"
concentration = label_power.artist_concentration(release_labels, real_artists, min_releases=50)
concentration.join(tagged.select("label_id", "major"), "label_id").groupBy("major").agg(
    F.count("*").alias("n_labels"), F.avg("hhi").alias("mean_hhi"), F.avg("top_artist_share").alias("mean_top_share")
).show()

## 5. Top producers and writers per label

In [ ]:
producers = label_power.top_contributors(release_labels, release_credits, config.PRODUCTION_ROLES, top_n=10)
writers = label_power.top_contributors(release_labels, release_credits, config.WRITING_ROLES, top_n=10)
producers.orderBy("label_id", "rank").show(20, truncate=False)

In [ ]:
# Check: the roles in config.PRODUCTION_ROLES / WRITING_ROLES were confirmed against these counts in T04.
release_credits.groupBy("role").count().orderBy(F.desc("count")).show(40, truncate=False)

`top_contributors` counts credits at both levels: release-level credits (whole release) and track-level credits (single tracks). The next cell shows how many releases have a writer credit at each level.

In [ ]:
writer_levels = label_power.credit_level_coverage(release_credits, config.WRITING_ROLES).toPandas()
writer_levels.pivot(index="year", columns="level", values="n_releases")

## 6. Concentration of the whole major tree

Pools every label in a major's tree, so a release on two Universal labels counts once. Compare with the per-label means in section 4: a tree has many artists, so its HHI is far lower than that of its single labels. `by_year` shows the trend; 2026 has fewer releases (Jan-Sep), which pushes HHI up.

In [ ]:
tree_concentration = label_power.group_concentration(release_labels, real_artists, tagged, group_col="major", min_releases=50)
tree_concentration.toPandas().sort_values("major")

In [ ]:
tree_by_year = label_power.group_concentration(
    release_labels, real_artists, tagged, group_col="major", min_releases=50, by_year=True
).toPandas().sort_values(["major", "year"])
viz.plot_lines(
    tree_by_year, "year", "hhi", "major",
    title="HHI of artist shares inside each label group, 2016-2026 (2026 is partial: Jan-Sep)",
)
tree_by_year[tree_by_year.year.isin([2016, 2025])]

## 7. Releases on labels with no parent information

"Independent" means "not linked to a major's tree in Discogs". A label with no parent and no children may still belong to a major whose contributors never filled in the link. This table counts distinct releases per year by the best class of their labels: `in_tree` (label has a parent or children) > `on_root_alone` > `self_released` > `unknown_label` (label id not in the labels table). `on_root_alone` is the pool where hidden major output could sit.

`self_released` holds the "Not On Label" placeholders (label 1818 and every label named "Not On Label ...", found by `self_released_label_ids`). They mean the artist released the music, not that a company did, and they hold 12.8-17.9% of each year's releases (the parent-coverage table below). Without this class, label 1818 counts as `in_tree` because its `parent_id` points to Stolen Riff Records (205754). That link is a data-entry error. A release on a placeholder and on a real label takes the real label's class.

In [ ]:
self_released = label_power.self_released_label_ids(labels)
parent_cov = label_power.parent_coverage(release_labels, tagged, self_released=self_released).toPandas()
parent_cov

## Decisions to carry into `main.ipynb`

Filled in from the full-data runs in `executed/`. Each line should end up as a sentence in the report's methodology or limitations.

- [x] Root labels of the majors: Universal Music Group 38404, Sony Corporation 353657 and Warner Music Group 2345 (`config.MAJOR_ROOT_IDS`). Universal and Warner sit under the nameless placeholder label 212, which is not followed (`config.PLACEHOLDER_LABEL_IDS`). The trees hold 8,080, 3,633 and 3,087 labels.
- [x] `max_depth`: 12. 2,425,097 labels reach a root and 978 do not; 665 of these stop at depth 12, most likely because of loops in the parent links.
- [x] Minimum size for the concentration measures: 50 release-artist pairs per label (`min_releases=50`), which keeps 267 Universal, 178 Sony, 156 Warner and 15,124 independent labels.
- [x] Role lists: producers and engineers are `config.PRODUCTION_ROLES` (Producer, Co-producer, Executive-Producer, Engineer, Mixed By, Mastered By, Recorded By, Remastered By, Lacquer Cut By); writers are `config.WRITING_ROLES` (Written-By, Written By, Composed By, Lyrics By, Music By, Songwriter). Checked against the role counts, where Written-By (1,805,372) and Producer (1,482,212) are the two most common roles.